# 03 — Khai Pha: Association Rules + Clustering
Apriori - FP-Growth - K-Means - Silhouette - PCA

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data.loader         import load_config
from src.mining.association  import AssociationMiner
from src.mining.clustering   import ClusterMiner
from src.evaluation.report   import Reporter
from src.visualization.plots import plot_association_rules, plot_elbow, plot_clusters

cfg      = load_config("../configs/params.yaml")
df_disc  = pd.read_parquet("../data/processed/yield_discrete.parquet")
df_fe    = pd.read_parquet("../data/processed/yield_features.parquet")
reporter = Reporter(cfg)
print("Data loaded OK")

## 3.1 Association Rules

In [ ]:
miner = AssociationMiner(cfg)
miner.fit(df_disc)

print("\nSo sanh Apriori vs FP-Growth:")
print(miner.compare_algorithms().to_string(index=False))

In [ ]:
reporter.print_top_rules(miner.rules_high_, n=10)

In [ ]:
plot_association_rules(miner.rules_, miner.rules_high_, cfg["paths"]["figures"])
plt.show()

In [ ]:
reporter.save_table(miner.compare_algorithms(), "association_compare.csv")
reporter.save_table(miner.top_rules(20), "top_rules.csv")
print("Tables saved")

## 3.2 Clustering K-Means

In [ ]:
clusterer = ClusterMiner(cfg)
clusterer.fit(df_fe)

In [ ]:
plot_elbow(clusterer.inertias_, cfg["paths"]["figures"])
plt.show()

In [ ]:
print(f"Silhouette Score: {clusterer.sil_score_:.4f}")
print(f"Davies-Bouldin Index: {clusterer.dbi_score_:.4f}")
print("\nPROFILE CAC CUM:")
reporter.print_cluster_profile(clusterer.profile_, clusterer.get_cluster_names())

In [ ]:
plot_clusters(clusterer.X_pca_, clusterer.labels_,
              clusterer.pca_var_, clusterer.profile_,
              cfg["paths"]["figures"])
plt.show()

In [ ]:
reporter.save_table(clusterer.profile_.reset_index(), "cluster_profile.csv")
print("Done")